<font size=10>**TASK 3 - ??**</font> <a class="anchor" id='title'></a> 

**Bachelor's in Data Science - NOVA IMS (25/26)**

**Project**: *Straining the great southern Melting Pot*

**Group 8**
- Beatriz Marques 20231605
- David Carrilho 20231693
- Duarte Fernandes 20231619
- Filipe Caçador 20231707
- Mariana Calais-Pedro 20231641

*«notebook description»*

**Question**: *??*

<font color = 'red'>**IR3 – Named Entity Recognition of Culinary and Spatial Themes**
Can we automatically extract dishes and locations from restaurant reviews using Named Entity Recognition (NER), and do the patterns of these entities reveal meaningful insights about cuisine types and dining experiences?
 
(Named Entity Recognition – focusing on dishes and locations, with optional analysis of entity transitions)

----

**IR3 – Dish Entity Networks and Cuisine Clusters**
 
Can we automatically extract dishes and related entities (e.g., locations) from restaurant reviews, and do the co-occurrence patterns of these entities form meaningful clusters that reveal cuisine types and cultural links? 

(NER + Co-occurrence Analysis + Clustering)</font>

<font color='#BFD72' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a> 
- [1. Imports](#1)
- [2. Data](#2)
- [3. Named Entity Rcognition](#3)
    - [3.1 Specific Data Preparation](#3_1)
    - [3.2 Model Implementation](#3_2)
    - [3.3 Model Evaluation](#3_3)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [1]:
import warnings
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')

In [2]:
import sys
import os
import pandas as pd
import spacy
from spacy import displacy
import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.model_selection import train_test_split

# Get the absolute path of the source_code folder
source_code_path = os.path.abspath('../source')

# Add the source_code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

from my_utils import *
from visualizations import *
from general_preprocessing import *

# <font color='#BFD72F' size=6>**2. Data**</font> <a class="anchor" id="2"></a>
  
[Back to TOC](#toc)

In [3]:
dataset_original = load_dataset('../data/02_atlanta_restaurant_slice_2023_translated.csv')

In [4]:
dataset_original.head()

,title,categoryName,website,url,reviewsCount,stars,text,is_chain,total_reviews_by_title,latitude,longitude,num_sentences,00_before_translating_cleaning,lang_langdetect,lang_langid,needs_translation,text_translated,text_for_pipeline
0,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,"One word amazing!! The red fish, halibut, fr...",False,3349,33.779814,-84.410451,3,"One word amazing!! The red fish, halibut, frie...",en,en,False,"One word amazing!! The red fish, halibut, frie...","One word amazing!! The red fish, halibut, frie..."
1,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,First time here and the food is great and the ...,False,3349,33.779814,-84.410451,1,First time here and the food is great and the ...,en,en,False,First time here and the food is great and the ...,First time here and the food is great and the ...
2,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,I recently had the pleasure of dining at Optim...,False,3349,33.779814,-84.410451,14,I recently had the pleasure of dining at Optim...,en,en,False,I recently had the pleasure of dining at Optim...,I recently had the pleasure of dining at Optim...
3,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,Beautiful atmosphere and delicious food. All o...,False,3349,33.779814,-84.410451,9,Beautiful atmosphere and delicious food . All ...,en,en,False,Beautiful atmosphere and delicious food . All ...,Beautiful atmosphere and delicious food . All ...
4,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,We had a wonderful dinner at the Optimist. Our...,False,3349,33.779814,-84.410451,3,We had a wonderful dinner at the Optimist . Ou...,en,en,False,We had a wonderful dinner at the Optimist . Ou...,We had a wonderful dinner at the Optimist . Ou...


In [5]:
dataset_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53565 entries, 0 to 53564
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   title                           53565 non-null  object 
 1   categoryName                    53565 non-null  object 
 2   website                         50599 non-null  object 
 3   url                             53565 non-null  object 
 4   reviewsCount                    53565 non-null  int64  
 5   stars                           53565 non-null  float64
 6   text                            53565 non-null  object 
 7   is_chain                        53565 non-null  bool   
 8   total_reviews_by_title          53565 non-null  int64  
 9   latitude                        53565 non-null  float64
 10  longitude                       53565 non-null  float64
 11  num_sentences                   53565 non-null  int64  
 12  00_before_translating_cleaning  

| 🏷️ **Column Name** | 📝 **Description** |
|:-------------------|:-------------------|
|**title** | Name of the restaurant |
|**categoryName** | Labels that describe the restaurant's cuisine type |
|**website** | URL of the restaurant's webpage |
|**url** | URL of the restaurant's Google Maps page |
|**reviewsCount** | Total number of reviews for the restaurant at the time of scraping |
|**stars** | Customer rating (1 to 5) |
|**text** | Text of the review |

In [6]:
dataset = dataset_original.copy()

# <font color='#BFD72F' size=6>**3. Named Entity Recognition**</font> <a class="anchor" id="3"></a>
  
[Back to TOC](#toc)

## <font color='#BFD72F' size=6>3.1 Specific Data Preparation</font> <a class="anchor" id="3_1"></a>
  
[Back to TOC](#toc)

In [7]:
# Tokenization and POS tagging using project pipeline
dataset["pos_list"] =\
      dataset["text_for_pipeline"].map(lambda content : main_pipeline(content,
                                                            print_output=False,
                                                            no_stopwords=False,
                                                            lowercase=False,
                                                            lemmatized=False,
                                                            no_punctuation=False,
                                                            pos_tags_list='pos_list'
                                                            ))

dataset["tokens"] =\
      dataset["text_for_pipeline"].map(lambda content : main_pipeline(content,
                                                            print_output=False,
                                                            no_stopwords=False,
                                                            lowercase=False,
                                                            lemmatized=False,
                                                            no_punctuation=False,
                                                            tokenized_output=True
                                                            ))


In [8]:
dataset['features'] = dataset.apply(lambda row: sent2features(row['tokens'], row['pos_list']), axis=1)

In [9]:
dataset["features"].sample(1, random_state=12).iloc[0]

[{'bias': 1.0,
  'word.lower()': 'always',
  'word[-3:]': 'ays',
  'word[-2:]': 'ys',
  'word.isupper()': False,
  'word.istitle()': True,
  'word.isdigit()': False,
  'postag': 'VBZ',
  'postag[:2]': 'VB',
  'BOS': True,
  '+1:word.lower()': 'a',
  '+1:word.istitle()': False,
  '+1:word.isupper()': False,
  '+1:postag': 'DT',
  '+1:postag[:2]': 'DT'},
 {'bias': 1.0,
  'word.lower()': 'a',
  'word[-3:]': 'a',
  'word[-2:]': 'a',
  'word.isupper()': False,
  'word.istitle()': False,
  'word.isdigit()': False,
  'postag': 'DT',
  'postag[:2]': 'DT',
  '-1:word.lower()': 'always',
  '-1:word.istitle()': True,
  '-1:word.isupper()': False,
  '-1:postag': 'VBZ',
  '-1:postag[:2]': 'VB',
  '+1:word.lower()': 'great',
  '+1:word.istitle()': False,
  '+1:word.isupper()': False,
  '+1:postag': 'JJ',
  '+1:postag[:2]': 'JJ'},
 {'bias': 1.0,
  'word.lower()': 'great',
  'word[-3:]': 'eat',
  'word[-2:]': 'at',
  'word.isupper()': False,
  'word.istitle()': False,
  'word.isdigit()': False,
  'pos

In [10]:
nlp = spacy.load("en_core_web_sm")

equivalence_table = {"PERSON":"-per",
                     "NORP":"-grp",
                     "FAC":"-fac",
                     "ORG":"-org",
                     "GPE":"-gpe",
                     "LOC":"-geo",
                     "DATE":"-date",
                     "TIME":"-tim",
                     "WORK_OF_ART":"-art",
                     "LAW":"-law",
                     "LANGUAGE":"-lang",
                     "EVENT":"eve",
                     "PRODUCT":"-prod",
                     "PERCENT":"-pct",
                     "MONEY":"-mon",
                     "QUANTITY":"-qty",
                     "CARDINAL":"-card",
                     "ORDINAL":"-ord",
                     "":""
                     }

In [11]:
dataset["ner_labels_custom"] = dataset.apply(
    lambda row: align_bio_to_custom_tokens(
        row["text_for_pipeline"], 
        row["tokens"],
        nlp,
        equivalence_table
    ),
    axis=1
)

In [12]:
# Sanity check
dataset[['text_for_pipeline','tokens','ner_labels_custom']].head()

,text_for_pipeline,tokens,ner_labels_custom
0,"One word amazing!! The red fish, halibut, frie...","[One, word, amazing, !, !, The, red, fish, ,, ...","[B-card, O, O, O, O, O, O, O, O, O, O, O, O, O..."
1,First time here and the food is great and the ...,"[First, time, here, and, the, food, is, great,...","[B-ord, O, O, O, O, O, O, O, O, O, O, O, O, O]"
2,I recently had the pleasure of dining at Optim...,"[I, recently, had, the, pleasure, of, dining, ...","[O, O, O, O, O, O, O, O, B-per, O, B-gpe, O, B..."
3,Beautiful atmosphere and delicious food . All ...,"[Beautiful, atmosphere, and, delicious, food, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
4,We had a wonderful dinner at the Optimist . Ou...,"[We, had, a, wonderful, dinner, at, the, Optim...","[O, O, O, O, O, O, O, B-grp, O, O, O, O, B-car..."


In [13]:
# # Expand rows so each token becomes its own row
# dataset["ner_labels"] = dataset.apply(
#     lambda row: align_bio(nlp(row["text_for_pipeline"]), row["tokens"]),
#     axis=1
# )

In [14]:
# dataset.head()

## <font color='#BFD72F' size=6>3.2 Model Implementation</font> <a class="anchor" id="3_2"></a>
  
[Back to TOC](#toc)

In [15]:
# Quick (q1) train-test split that we can use to test our classifier
X_train_q1, X_test_q1, y_train_q1, y_test_q1 = train_test_split(
                                                        dataset["features"], 
                                                        dataset["ner_labels_custom"],
                                                        test_size=0.2,
                                                        random_state=39
                                                        )

train_index_q1 = list(X_train_q1.index)
test_index_q1 = list(X_test_q1.index)

In [16]:
# CRF model -> conditional random field model

crf = sklearn_crfsuite.CRF(algorithm='lbfgs', 
                           c1=0.1, 
                           c2=0.1, 
                           max_iterations=100, 
                           all_possible_transitions=True,
                           verbose=True 
                           )
try:
    crf.fit(X_train_q1, y_train_q1)
except AttributeError:
    pass

loading training data to CRFsuite: 100%|██████████| 42852/42852 [00:12<00:00, 3387.85it/s]



Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 1
0....1....2....3....4....5....6....7....8....9....10
Number of features: 130043
Seconds required: 2.436

L-BFGS optimization
c1: 0.100000
c2: 0.100000
num_memories: 6
max_iterations: 100
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

Iter 1   time=12.68 loss=3415998.67 active=129248 feature_norm=1.00
Iter 2   time=12.89 loss=2154677.84 active=129005 feature_norm=14.94
Iter 3   time=6.32  loss=2009793.88 active=119008 feature_norm=14.09
Iter 4   time=104.75 loss=418400.92 active=80135 feature_norm=5.76
Iter 5   time=57.41 loss=411583.43 active=93027 feature_norm=5.97
Iter 6   time=39.35 loss=401732.96 active=94943 feature_norm=5.95
Iter 7   time=44.30 loss=368796.18 active=93845 feature_norm=5.83
Iter 8   time=57.10 loss=356064.43 active=99382 feature_norm=6.29
Iter 9   time=19.56 loss=339197.06 active=94048 feature_norm=

In [17]:
# keep tokens for inspection
X_test_tokens_q1 = dataset["tokens"].loc[test_index_q1]
y_pred_q1 = crf.predict(X_test_q1)

In [18]:
# # save model
# import joblib

# models_dir = os.path.join('..','models')
# os.makedirs(models_dir, exist_ok=True)
# joblib.dump(crf, os.path.join(models_dir, 'crf_ner_template.pkl'))

## <font color='#BFD72F' size=6>3.3 Model Evaluation</font> <a class="anchor" id="3_3"></a>
  
[Back to TOC](#toc)

In [19]:
labels = list(crf.classes_)
labels.remove("O")

round(metrics.flat_f1_score(y_test_q1, y_pred_q1,average='weighted', labels=labels), 3)

0.745

In [20]:
sorted_labels = sorted(
    labels,
    key=lambda name: (name[1:], name[0])
)

print(sklearn_crfsuite.metrics.flat_classification_report(y_test_q1, y_pred_q1, labels=sorted_labels, digits=3))

              precision    recall  f1-score   support

       B-art      0.405     0.195     0.264        87
       I-art      0.389     0.176     0.243       119
      B-card      0.836     0.863     0.849      2244
      I-card      0.813     0.713     0.759       268
      B-date      0.868     0.836     0.852      1387
      I-date      0.862     0.842     0.852      1191
       B-fac      0.500     0.348     0.410       115
       I-fac      0.466     0.378     0.417       143
       B-geo      0.600     0.480     0.533       100
       I-geo      0.333     0.227     0.270        44
       B-gpe      0.795     0.720     0.756       972
       I-gpe      0.792     0.787     0.789       169
       B-grp      0.817     0.792     0.804       965
       I-grp      0.792     0.500     0.613        38
      B-lang      0.467     0.467     0.467        15
       B-law      0.500     0.158     0.240        19
       I-law      0.571     0.211     0.308        38
       B-mon      0.831    

In [21]:
dataset

,title,categoryName,website,url,reviewsCount,stars,text,is_chain,total_reviews_by_title,latitude,...,00_before_translating_cleaning,lang_langdetect,lang_langid,needs_translation,text_translated,text_for_pipeline,pos_list,tokens,features,ner_labels_custom
0,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,"One word amazing!! The red fish, halibut, fr...",False,3349,33.779814,...,"One word amazing!! The red fish, halibut, frie...",en,en,False,"One word amazing!! The red fish, halibut, frie...","One word amazing!! The red fish, halibut, frie...","[CD, NN, NN, ., ., DT, JJ, NN, ,, NN, ,, VBN, ...","[One, word, amazing, !, !, The, red, fish, ,, ...","[{'bias': 1.0, 'word.lower()': 'one', 'word[-3...","[B-card, O, O, O, O, O, O, O, O, O, O, O, O, O..."
1,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,First time here and the food is great and the ...,False,3349,33.779814,...,First time here and the food is great and the ...,en,en,False,First time here and the food is great and the ...,First time here and the food is great and the ...,"[JJ, NN, RB, CC, DT, NN, VBZ, JJ, CC, DT, NN, ...","[First, time, here, and, the, food, is, great,...","[{'bias': 1.0, 'word.lower()': 'first', 'word[...","[B-ord, O, O, O, O, O, O, O, O, O, O, O, O, O]"
2,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,I recently had the pleasure of dining at Optim...,False,3349,33.779814,...,I recently had the pleasure of dining at Optim...,en,en,False,I recently had the pleasure of dining at Optim...,I recently had the pleasure of dining at Optim...,"[PRP, RB, VBD, DT, NN, IN, VBG, IN, NNP, IN, N...","[I, recently, had, the, pleasure, of, dining, ...","[{'bias': 1.0, 'word.lower()': 'i', 'word[-3:]...","[O, O, O, O, O, O, O, O, B-per, O, B-gpe, O, B..."
3,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,Beautiful atmosphere and delicious food. All o...,False,3349,33.779814,...,Beautiful atmosphere and delicious food . All ...,en,en,False,Beautiful atmosphere and delicious food . All ...,Beautiful atmosphere and delicious food . All ...,"[NNP, RB, CC, JJ, NN, ., DT, IN, DT, NN, NN, N...","[Beautiful, atmosphere, and, delicious, food, ...","[{'bias': 1.0, 'word.lower()': 'beautiful', 'w...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
4,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,We had a wonderful dinner at the Optimist. Our...,False,3349,33.779814,...,We had a wonderful dinner at the Optimist . Ou...,en,en,False,We had a wonderful dinner at the Optimist . Ou...,We had a wonderful dinner at the Optimist . Ou...,"[PRP, VBD, DT, JJ, NN, IN, DT, NNP, ., PRP$, N...","[We, had, a, wonderful, dinner, at, the, Optim...","[{'bias': 1.0, 'word.lower()': 'we', 'word[-3:...","[O, O, O, O, O, O, O, B-grp, O, O, O, O, B-car..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53560,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Friday night dinner was Chicken Francaise. Jor...,False,449,33.953384,...,Friday night dinner was Chicken Francaise . Jo...,en,en,False,Friday night dinner was Chicken Francaise . Jo...,Friday night dinner was Chicken Francaise . Jo...,"[NNP, NN, NN, VBD, NNP, NNP, ., NNP, VBD, DT, ...","[Friday, night, dinner, was, Chicken, Francais...","[{'bias': 1.0, 'word.lower()': 'friday', 'word...","[B-tim, I-tim, O, O, B-per, I-per, O, B-per, O..."
53561,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Great dinner.... Yay Jordan

### <font color='#BFD72F' size=6>4. Third Task</font> <a class="anchor" id="4"></a>
  
[Back to TOC](#toc)

In [22]:
#pip install louvain

In [35]:
import pandas as pd
import networkx as nx
from collections import defaultdict

df = dataset.copy()

# ---------------------------------------------------
# 1. Decode BIO NER tags into entity spans
# ---------------------------------------------------

def extract_entities(tokens, labels):
    """
    Extract entities from BIO-tagged tokens.
    Returns a list of (entity_text, entity_type)
    """
    entities = []
    current = []
    current_type = None

    for tok, tag in zip(tokens, labels):
        if tag.startswith("B-"):
            # start of a new entity
            if current:
                entities.append((" ".join(current), current_type))
            current = [tok]
            current_type = tag[2:]  # remove B-
        elif tag.startswith("I-") and current:
            current.append(tok)
        else:
            # Outside tag: close existing entity
            if current:
                entities.append((" ".join(current), current_type))
                current = []
                current_type = None

    # flush last entity
    if current:
        entities.append((" ".join(current), current_type))

    return entities

# ---------------------------------------------------
# 2. Extract DISH + LOCATION + CUISINE entities
# ---------------------------------------------------

# Adjust these to match your real labels:
DISH_TAGS = {"food", "dish", "meal"}
LOC_TAGS = {"loc", "geo", "car", "gpe"}        
CUISINE_TAGS = {"grp", "norp"}                 # grp might include ethnicities

def classify_entity(ent_type):
    et = ent_type.lower()
    if et in DISH_TAGS:
        return "DISH"
    elif et in LOC_TAGS:
        return "LOC"
    elif et in CUISINE_TAGS:
        return "CUISINE"
    else:
        return None  # ignore others

# ---------------------------------------------------
# 3. Build co-occurrence graph
# ---------------------------------------------------

G = nx.Graph()

for i, row in df.iterrows():
    tokens = row["tokens"]
    labels = row["ner_labels_custom"]

    entities = extract_entities(tokens, labels)

    # Keep only selected entity types
    filtered = [
        (text_for_pipeline.lower(), classify_entity(tp))
        for text_for_pipeline, tp in entities
        if classify_entity(tp) is not None
    ]

    # Add nodes
    for ent_text, ent_type in filtered:
        if ent_text not in G.nodes:
            G.add_node(ent_text, type=ent_type)

    # Add edges for co-occurrence inside the same review
    for i in range(len(filtered)):
        for j in range(i+1, len(filtered)):
            a = filtered[i][0]
            b = filtered[j][0]

            if G.has_edge(a, b):
                G[a][b]['weight'] += 1
            else:
                G.add_edge(a, b, weight=1)

# ---------------------------------------------------
# 4. Community Detection (Louvain)
# ---------------------------------------------------

try:
    import community  # python-louvain
    partition = community.best_partition(G, weight="weight")
except ImportError:
    print("Install python-louvain: pip install python-louvain")
    partition = {node: 0 for node in G.nodes()}  # fallback

# ---------------------------------------------------
# 5. Convert clusters to DataFrame
# ---------------------------------------------------

# ---------------------------------------------------
# 5. Infer Names and Create DataFrame
# ---------------------------------------------------

def infer_cluster_name(nodes, graph):
    """
    Infers a name for a cluster based on the importance (weighted degree)
    of its constituent nodes.
    Priority: CUISINE > LOC > DISH
    """
    # dictionaries to hold total weight for each entity in the cluster
    cuisine_scores = defaultdict(int)
    loc_scores = defaultdict(int)
    dish_scores = defaultdict(int)
    
    for node in nodes:
        # Get the type we assigned in Step 3
        if not graph.has_node(node): 
            continue
            
        attrs = graph.nodes[node]
        ent_type = attrs.get('type')
        
        # Calculate importance: The weighted degree of the node within the whole graph
        # (or you could restrict this to degree within the subgraph)
        score = graph.degree(node, weight='weight')
        
        if ent_type == "CUISINE":
            cuisine_scores[node] += score
        elif ent_type == "LOC":
            loc_scores[node] += score
        elif ent_type == "DISH":
            dish_scores[node] += score
            
    # Heuristic 1: Dominant Cuisine (Highest weighted cuisine node)
    if cuisine_scores:
        best_cuisine = max(cuisine_scores, key=cuisine_scores.get)
        return f"{best_cuisine.title()} Cuisine"
    
    # Heuristic 2: Dominant Location (Highest weighted location node)
    if loc_scores:
        best_loc = max(loc_scores, key=loc_scores.get)
        return f"{best_loc.title()} Location"
    
    # Heuristic 3: Top Dishes (Top 2 highest weighted dishes)
    if dish_scores:
        # Sort dishes by score descending
        sorted_dishes = sorted(dish_scores, key=dish_scores.get, reverse=True)
        top_two = [d.title() for d in sorted_dishes[:2]]
        return f"{' & '.join(top_two)} Cluster"
        
    return "General Cluster"

# Group nodes by cluster ID
clusters = defaultdict(list)
for node, cluster_id in partition.items():
    clusters[cluster_id].append(node)

# Create the dataframe with inferred names
cluster_data = []
for cid, nodes in clusters.items():
    name = infer_cluster_name(nodes, G)
    cluster_data.append({
        "Cluster ID": cid,
        "Inferred Name": name,
        "Entities": nodes,
        "Size": len(nodes)
    })

clusters_df = pd.DataFrame(cluster_data)

# Reorder columns for readability
clusters_df = clusters_df[["Cluster ID", "Inferred Name", "Size", "Entities"]]

print(clusters_df.head(10))

# clusters = defaultdict(list)
# for node, cluster_id in partition.items():
#     clusters[cluster_id].append(node)

# clusters_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in clusters.items()]))

# clusters_df.head()


   Cluster ID            Inferred Name  Size  \
0          16          Italian Cuisine   158   
1           1         Optimist Cuisine     3   
2           2  Southern India Location     1   
3           3          Kazhan Location     1   
4           4           Zuppa Location     1   
5          74           Cheese Cuisine    61   
6           8           French Cuisine   127   
7           5          Mexican Cuisine   152   
8           9     Veracruzana Location     1   
9          45          Chinese Cuisine    70   

                                            Entities  
0  [atlanta, italian, nino, alfredo, italy, olive...  
1                  [optimist, round island, plancha]  
2                                   [southern india]  
3                                           [kazhan]  
4                                            [zuppa]  
5  [kinda, risotto, wild mushrooms, cheese, steak...  
6  [new york, georgia, french, chicago, new jerse...  
7  [mexican, el valle, texan, n

In [36]:
clusters_df.head(10)

,Cluster ID,Inferred Name,Size,Entities
0,16,Italian Cuisine,158,"[atlanta, italian, nino, alfredo, italy, olive..."
1,1,Optimist Cuisine,3,"[optimist, round island, plancha]"
2,2,Southern India Location,1,[southern india]
3,3,Kazhan Location,1,[kazhan]
4,4,Zuppa Location,1,[zuppa]
5,74,Cheese Cuisine,61,"[kinda, risotto, wild mushrooms, cheese, steak..."
6,8,French Cuisine,127,"[new york, georgia, french, chicago, new jerse..."
7,5,Mexican Cuisine,152,"[mexican, el valle, texan, noche, tostitos, el..."
8,9,Veracruzana Location,1,[veracruzana]
9,45,Chinese Cuisine,70,"[central american, asian, honduran, thai, kore..."


In [24]:
#pip install pyvis

In [25]:
#pip install sentence_transformers

In [26]:
#pip install hf_xet

In [27]:
#pip install jinja2

In [ ]:
import pandas as pd
import networkx as nx
from collections import defaultdict, Counter
from sklearn.cluster import AgglomerativeClustering
from sentence_transformers import SentenceTransformer
from pyvis.network import Network
import numpy as np
import community  # python-louvain
from jinja2 import Template
import pkgutil

# -----------------------------------------------------------
# 1. BIO Decoder
# -----------------------------------------------------------
def extract_bio_entities(tokens, labels):
    entities = []
    current_tokens = []
    current_type = None

    for tok, tag in zip(tokens, labels):
        if tag.startswith("B-"):
            if current_tokens:
                entities.append((" ".join(current_tokens), current_type))
            current_tokens = [tok]
            current_type = tag[2:]
        elif tag.startswith("I-") and current_tokens:
            current_tokens.append(tok)
        else:
            if current_tokens:
                entities.append((" ".join(current_tokens), current_type))
            current_tokens = []
            current_type = None

    if current_tokens:
        entities.append((" ".join(current_tokens), current_type))

    return entities

# -----------------------------------------------------------
# 2. Entity classification
# -----------------------------------------------------------
# Removed 'ord' to fix the "1st" node issue
DISH_TAGS = {"food", "dish", "meal"} 
LOC_TAGS = {"loc", "geo", "gpe", "car"}
CUISINE_TAGS = {"grp", "norp"}

def classify_entity(ent_type):
    if ent_type is None:
        return None
    et = ent_type.lower()
    if et in DISH_TAGS:
        return "DISH"
    if et in LOC_TAGS:
        return "LOC"
    if et in CUISINE_TAGS:
        return "CUISINE"
    return None

# -----------------------------------------------------------
# 3. Embedding-based dish merging
# -----------------------------------------------------------
def merge_similar_dishes(dish_list, threshold=0.85): # Increased threshold slightly for stricter matching
    if len(dish_list) == 0:
        return {}

    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(dish_list)

    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric='cosine',
        linkage='average',
        distance_threshold=1 - threshold
    )

    labels = clustering.fit_predict(embeddings)

    merged = defaultdict(list)
    for dish, cluster_id in zip(dish_list, labels):
        merged[cluster_id].append(dish)

    mapping = {}
    for cid, group in merged.items():
        canonical = min(group, key=len)
        for g in group:
            mapping[g] = canonical
    
    return mapping

# -----------------------------------------------------------
# 4. Extract and Normalize Dishes
# -----------------------------------------------------------
all_dishes = []
for idx, row in df.iterrows():
    ents = extract_bio_entities(row["tokens"], row["ner_labels_custom"])
    for text_for_pipeline, tp in ents:
        if classify_entity(tp) == "DISH":
            all_dishes.append(text_for_pipeline.lower())

dish_freq = Counter(all_dishes)
dish_mapping = merge_similar_dishes(list(dish_freq.keys()))

def normalize_dish(d):
    return dish_mapping.get(d, d)

# -----------------------------------------------------------
# 5. Build Graph & Sparsify
# -----------------------------------------------------------
# Added stoplist to remove generic noise
STOPLIST = {"atlanta", "restaurant", "place", "spot", "dinner", "lunch", "menu", "food", "meal", "area", "town", "city", "marietta", "georgia"}
MIN_NODE_FREQ = 5 
G_full = nx.Graph()
entity_frequency = Counter()

# Count frequencies
for idx, row in df.iterrows():
    ents = extract_bio_entities(row["tokens"], row["ner_labels_custom"])
    for text_for_pipeline, tp in ents:
        etype = classify_entity(tp)
        if etype:
            text_norm = text_for_pipeline.lower()
            if etype == "DISH":
                text_norm = normalize_dish(text_norm)
            if text_norm not in STOPLIST:
                entity_frequency[text_norm] += 1

# Build full graph first
for idx, row in df.iterrows():
    ents = extract_bio_entities(row["tokens"], row["ner_labels_custom"])
    filtered = []
    for text_for_pipeline, tp in ents:
        etype = classify_entity(tp)
        if not etype: continue

        text_norm = text_for_pipeline.lower()
        if text_norm in STOPLIST: continue # Skip stop words
        
        if etype == "DISH":
            text_norm = normalize_dish(text_norm)

        if entity_frequency[text_norm] >= MIN_NODE_FREQ:
            filtered.append((text_norm, etype))

    # Add nodes and edges
    for ent_text, ent_type in filtered:
        if not G_full.has_node(ent_text):
            G_full.add_node(ent_text, type=ent_type)

    for i in range(len(filtered)):
        for j in range(i+1, len(filtered)):
            a, _ = filtered[i]
            b, _ = filtered[j]
            if G_full.has_edge(a, b):
                G_full[a][b]['weight'] += 1
            else:
                G_full.add_edge(a, b, weight=1)

# --- NEW: Sparsify Graph (Keep Top-K Edges) ---
def sparsify_graph(graph, k=5):
    """
    For every node, keep only the top k strongest connections.
    This removes the 'hairball' effect while keeping structure.
    """
    new_edges = set()
    for n in graph.nodes():
        # Get all neighbors with weights
        neighbors = graph[n].items()
        # Sort by weight (descending)
        sorted_neighbors = sorted(neighbors, key=lambda x: x[1]['weight'], reverse=True)
        # Keep top k
        for nbr, data in sorted_neighbors[:k]:
            # Sort tuple to ensure undirected edge uniqueness
            edge = tuple(sorted((n, nbr)))
            new_edges.add((edge[0], edge[1], data['weight']))
            
    G_clean = nx.Graph()
    G_clean.add_nodes_from(graph.nodes(data=True)) # Keep all nodes
    
    # Add back only the strong edges
    for u, v, w in new_edges:
        G_clean.add_edge(u, v, weight=w)
        
    # Remove isolated nodes that lost all connections
    G_clean.remove_nodes_from(list(nx.isolates(G_clean)))
    return G_clean

G = sparsify_graph(G_full, k=5)  # Keep top 5 edges per node

# -----------------------------------------------------------
# 6. Louvain Clustering
# -----------------------------------------------------------
partition = community.best_partition(G, weight="weight")

clusters = defaultdict(list)
for node, cid in partition.items():
    clusters[cid].append(node)

# -----------------------------------------------------------
# 7. Smart Cluster Naming
# -----------------------------------------------------------
def infer_cluster_name(nodes, graph):
    cuisine_scores = defaultdict(int)
    loc_scores = defaultdict(int)
    dish_scores = defaultdict(int)
    
    for node in nodes:
        if not graph.has_node(node): continue
        attrs = graph.nodes[node]
        ent_type = attrs.get('type')
        # Use simple degree for naming within the sparsified graph
        score = graph.degree(node, weight='weight')
        
        if ent_type == "CUISINE":
            cuisine_scores[node] += score
        elif ent_type == "LOC":
            loc_scores[node] += score
        elif ent_type == "DISH":
            dish_scores[node] += score
            
    if cuisine_scores:
        best = max(cuisine_scores, key=cuisine_scores.get)
        return f"{best.title()} Cuisine"
    if loc_scores:
        best = max(loc_scores, key=loc_scores.get)
        return f"{best.title()} Location"
    if dish_scores:
        sorted_dishes = sorted(dish_scores, key=dish_scores.get, reverse=True)
        top_two = [d.title() for d in sorted_dishes[:2]]
        return f"{' & '.join(top_two)} Cluster"
        
    return "Mixed Cluster"

named_clusters = {
    cid: {
        "name": infer_cluster_name(nodes, G),
        "nodes": nodes
    }
    for cid, nodes in clusters.items()
}

# -----------------------------------------------------------
# 8. Interactive Graph Visualization (Sorted & Layered)
# -----------------------------------------------------------

net = Network(
    height="800px",
    width="100%",
    bgcolor="#ffffff",
    font_color="#333333",
    select_menu=True
)

# Pre-calculate node statistics
degrees = dict(G.degree(weight='weight'))
max_degree = max(degrees.values()) if degrees else 1

# --- Threshold Calculation ---
all_weights = [d['weight'] for u, v, d in G.edges(data=True)]
# Let's lower the bar slightly to top 10% to ensure we see them clearly
strong_threshold = np.percentile(all_weights, 90) if all_weights else 0

print(f"Highlighting edges with weight >= {strong_threshold}")

# Add Nodes
for node, data in G.nodes(data=True):
    ntype = data.get("type", "UNKNOWN")
    cluster_id = partition.get(node)
    
    group_name = named_clusters[cluster_id]["name"] if cluster_id in named_clusters else "Unknown"
    
    size = 10 + np.log(degrees.get(node, 1) + 1) * 8
    title_html = f"<b>{node.title()}</b><br>Type: {ntype}<br>Conn: {degrees.get(node,0)}"

    net.add_node(
        node, 
        label=node, 
        title=title_html,
        value=size,
        group=group_name,
        borderWidth=1
    )

# --- KEY FIX: Sort edges so WEAK are drawn first, STRONG last (on top) ---
edges_to_draw = []
for u, v, w in G.edges(data=True):
    weight = w['weight']
    is_strong = weight >= strong_threshold
    edges_to_draw.append((u, v, weight, is_strong))

# Sort: False (Weak) comes before True (Strong)
edges_to_draw.sort(key=lambda x: x[2]) 

for u, v, weight, is_strong in edges_to_draw:
    if is_strong:
        # STRONG: Bright Red, Thick, Opaque, Drawn Last
        net.add_edge(u, v, width=4, color={'color': '#ff0000', 'opacity': 1.0, 'inherit': False})
    else:
        # WEAK: Gray, Thin, Transparent, Drawn First
        net.add_edge(u, v, width=1, color={'color': '#cccccc', 'opacity': 0.2, 'inherit': False})

# Physics Layout
net.set_options("""
{
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -80,
      "centralGravity": 0.005,
      "springLength": 150,
      "springConstant": 0.05,
      "damping": 0.9
    },
    "maxVelocity": 40,
    "solver": "forceAtlas2Based",
    "stabilization": {
      "enabled": true,
      "iterations": 1000
    }
  },
  "edges": {
    "smooth": false
  }
}
""")

# Fix Template
data = pkgutil.get_data("pyvis", "templates/template.html")
if data is None:
    raise RuntimeError("PyVis template not found.")
Network.template = Template(data.decode("utf-8"))

output_file = "dish_entity_network_fixed_red.html"
net.write_html(output_file)
print(f"✔ Network saved to {output_file}")

Highlighting edges with weight >= 2.0
--- EXAMPLES OF STRONG CONNECTIONS (Search for these nodes!) ---
★ Strong Edge created between: [italian] and [peachtree city] (Weight: 4)
★ Strong Edge created between: [italian] and [parmesan] (Weight: 2)
★ Strong Edge created between: [italian] and [jersey] (Weight: 2)
★ Strong Edge created between: [italian] and [american] (Weight: 9)
★ Strong Edge created between: [italian] and [smyrna] (Weight: 3)
✔ Saved to dish_network_neon_test.html -- Please open this specific file!


In [69]:
import webbrowser

webbrowser.open("dish_entity_network_fixed_red.html")


True